# 2. Tracking: ByteTrack on VisDrone-MOT val, scored with TrackEval
Full-precision ONNX vs INT8 ONNX detector, cars and pedestrians.

In [1]:
import sys, time, shutil, zipfile, subprocess, configparser, numpy as np, pandas as pd, cv2
from pathlib import Path
from ultralytics import YOLO
W = Path('runs/v8n_640_s0/weights')
BENCH = {'VisDroneCar': (4, 3), 'VisDronePedestrian': (1, 0)}   # name: (annotation category, detector class id)
TE = Path('track_eval').resolve()

## Unpack data
Download `VisDrone2019-MOT-val.zip` (1.48 GB, link in the VisDrone-Dataset repo) into `data/mot/` first.

In [2]:
Z = Path('data/mot/VisDrone2019-MOT-val.zip')
if Z.exists() and not Path('data/mot/VisDrone2019-MOT-val').exists(): zipfile.ZipFile(Z).extractall('data/mot')
SRC = Path('data/mot/VisDrone2019-MOT-val')
seqs = sorted(p.stem for p in (SRC/'annotations').glob('*.txt')); print(len(seqs), 'sequences'); print(open(SRC/'annotations'/f'{seqs[0]}.txt').readline())

7 sequences
102,0,38,666,71,88,1,1,1,0



## Convert ground truth to TrackEval layout
Annotation columns: frame, id, left, top, w, h, score, category, truncation, occlusion. Keep score 1 rows of one category; class column set to 1.

In [3]:
frames = lambda s: sorted((SRC/'sequences'/s).glob('*.jpg'))
for bench, (cat, _) in BENCH.items():
    gs = f'{bench}-val'
    for s in seqs:
        a = np.loadtxt(SRC/'annotations'/f'{s}.txt', delimiter=',', ndmin=2)
        a = a[(a[:, 7] == cat) & (a[:, 6] == 1)]; a = a[np.lexsort((a[:, 1], a[:, 0]))]
        d = TE/'gt'/gs/s; (d/'gt').mkdir(parents=True, exist_ok=True)
        np.savetxt(d/'gt/gt.txt', np.c_[a[:, :6], np.ones((len(a), 3))], fmt='%d', delimiter=',')
        h, w = cv2.imread(str(frames(s)[0])).shape[:2]
        (d/'seqinfo.ini').write_text(f'[Sequence]\nname={s}\nimDir=img1\nframeRate=30\nseqLength={len(frames(s))}\nimWidth={w}\nimHeight={h}\nimExt=.jpg\n')
    (TE/'gt/seqmaps').mkdir(parents=True, exist_ok=True)
    (TE/'gt/seqmaps'/f'{gs}.txt').write_text('name\n' + '\n'.join(seqs) + '\n')

## TrackEval helper, and the perfect-tracker check
A copy of the ground truth must score 100 on HOTA, MOTA and IDF1 before any real number is trusted.

In [4]:
def score(bench, trackers):
    subprocess.run([sys.executable, 'scripts/run_mot_challenge.py', '--BENCHMARK', bench, '--SPLIT_TO_EVAL', 'val',
                    '--GT_FOLDER', str(TE/'gt'), '--TRACKERS_FOLDER', str(TE/'trackers'), '--TRACKERS_TO_EVAL', *trackers,
                    '--METRICS', 'HOTA', 'CLEAR', 'Identity', '--DO_PREPROC', 'False', '--USE_PARALLEL', 'False',
                    '--PRINT_CONFIG', 'False'], cwd='TrackEval', check=True, capture_output=True)
    out = {}
    for t in trackers:   # TrackEval's MOT loader reports every class under 'pedestrian'
        h, v = (l.split() for l in (TE/'trackers'/f'{bench}-val'/t/'pedestrian_summary.txt').read_text().splitlines()[:2])
        d = dict(zip(h, v)); out[t] = {k: float(d[k]) for k in ('HOTA', 'MOTA', 'IDF1')}
    return out

for bench in BENCH:
    dst = TE/'trackers'/f'{bench}-val/gt_copy/data'; dst.mkdir(parents=True, exist_ok=True)
    for s in seqs: shutil.copy(TE/'gt'/f'{bench}-val'/s/'gt/gt.txt', dst/f'{s}.txt')
    print(bench, score(bench, ['gt_copy']))

VisDroneCar {'gt_copy': {'HOTA': 100.0, 'MOTA': 100.0, 'IDF1': 100.0}}


VisDronePedestrian {'gt_copy': {'HOTA': 100.0, 'MOTA': 100.0, 'IDF1': 100.0}}


## ByteTrack with each detector
Fresh model per sequence, frames fed in order with `persist=True`. Tracker named explicitly (the Ultralytics default is now TrackTrack). CPU timing included.

In [5]:
def track(weights, name, max_frames=None):
    n, t = 0, 0.0
    for s in seqs:
        model = YOLO(weights, task='detect'); fr = frames(s)[:max_frames]; rows = {b: [] for b in BENCH}; t0 = time.perf_counter()
        for i, f in enumerate(fr, 1):
            r = model.track(str(f), persist=True, tracker='bytetrack.yaml', classes=[3, 0], device='cpu', imgsz=640, conf=0.1, verbose=False)[0]
            if r.boxes.id is None: continue
            for (x1, y1, x2, y2), k, c, p in zip(r.boxes.xyxy.numpy(), r.boxes.id.numpy().astype(int), r.boxes.cls.numpy().astype(int), r.boxes.conf.numpy()):
                for b, (_, cid) in BENCH.items():
                    if c == cid: rows[b].append(f'{i},{k},{x1:.2f},{y1:.2f},{x2-x1:.2f},{y2-y1:.2f},{p:.4f},-1,-1,-1')
        t += time.perf_counter() - t0; n += len(fr)
        for b, r_ in rows.items():
            d = TE/'trackers'/f'{b}-val'/name/'data'; d.mkdir(parents=True, exist_ok=True); (d/f'{s}.txt').write_text('\n'.join(r_) + '\n' if r_ else '')
    return n / t

# 50-frame check that the ONNX models work in track mode, then the full run
print('fps (test)', track(W/'best_int8.onnx', 'test', max_frames=50)); shutil.rmtree(TE/'trackers', ignore_errors=True)
fps = {'onnx_fp32': track(W/'best_fp32.onnx', 'onnx_fp32'), 'onnx_int8': track(W/'best_int8.onnx', 'onnx_int8')}; fps

Loading runs\v8n_640_s0\weights\best_int8.onnx for ONNX Runtime inference...


Using ONNX Runtime 1.32.0 with CPUExecutionProvider


Loading runs\v8n_640_s0\weights\best_int8.onnx for ONNX Runtime inference...


Using ONNX Runtime 1.32.0 with CPUExecutionProvider


Loading runs\v8n_640_s0\weights\best_int8.onnx for ONNX Runtime inference...


Using ONNX Runtime 1.32.0 with CPUExecutionProvider


... (33 outputs left out)

Using ONNX Runtime 1.32.0 with CPUExecutionProvider


Loading runs\v8n_640_s0\weights\best_int8.onnx for ONNX Runtime inference...


Using ONNX Runtime 1.32.0 with CPUExecutionProvider


{'onnx_fp32': 13.861314625869099, 'onnx_int8': 10.35080879330459}

## Score and write `reports/tracking_metrics.csv`

In [6]:
rows = []
for bench in BENCH:
    for k, v in score(bench, ['onnx_fp32', 'onnx_int8']).items(): rows.append({'class': bench[8:].lower(), 'detector': k, **v, 'fps_cpu': fps[k]})
res = pd.DataFrame(rows); res.to_csv('reports/tracking_metrics.csv', index=False); res

,class,detector,HOTA,MOTA,IDF1,fps_cpu
0,car,onnx_fp32,44.689,27.237,50.907,13.861315
1,car,onnx_int8,43.849,27.639,51.353,10.350809
2,pedestrian,onnx_fp32,31.500,18.991,37.909,13.861315
3,pedestrian,onnx_int8,30.501,18.708,36.826,10.350809


## Demo video (one sequence, boxes and ids)

In [7]:
s = seqs[0]; model = YOLO(W/'best_fp32.onnx', task='detect'); fr = frames(s)
h, w = cv2.imread(str(fr[0])).shape[:2]; vw = cv2.VideoWriter('reports/demo.mp4', cv2.VideoWriter_fourcc(*'mp4v'), 15, (w, h))
for f in fr[:300]: vw.write(model.track(str(f), persist=True, tracker='bytetrack.yaml', classes=[3, 0], device='cpu', conf=0.1, verbose=False)[0].plot())
vw.release()

Loading runs\v8n_640_s0\weights\best_fp32.onnx for ONNX Runtime inference...


Using ONNX Runtime 1.32.0 with CPUExecutionProvider
